In [1]:
import json
import pandas as pd

BATCH_SIZE = 50
IS_DEBUG = True
DEBUG_QUESTION_SIZE = 10
LLM_MODEL = "gpt-4.1"
DATASET_NAME = "hotpotQA"
KNOWLEDGE_GRAPH_PATH = 'outputs/knowledge_graphs'

## 1. Loading HotpotQA Dataset

In [2]:
train_hotpot_qa_path = r'C:\Users\Sudheera\Documents\Phd\LLMs_and_KGs\hotpotQA\hotpot_train_v1.1.json'
test_hotpot_qa_path = r'C:\Users\Sudheera\Documents\Phd\LLMs_and_KGs\hotpotQA\hotpot_test_v1.1.json'

In [3]:
with open(train_hotpot_qa_path, 'r', encoding='utf-8') as f:
    train_hotpot_qa_json = json.load(f)

with open(test_hotpot_qa_path, 'r', encoding='utf-8') as f:
    test_hotpot_qa_json = json.load(f)

In [4]:
train_hotpot_qa_json[0]

{'supporting_facts': [["Arthur's Magazine", 0], ['First for Women', 0]],
 'level': 'medium',
 'question': "Which magazine was started first Arthur's Magazine or First for Women?",
 'context': [['Radio City (Indian radio station)',
   ["Radio City is India's first private FM radio station and was started on 3 July 2001.",
    ' It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003).',
    ' It plays Hindi, English and regional songs.',
    ' It was launched in Hyderabad in March 2006, in Chennai on 7 July 2006 and in Visakhapatnam October 2007.',
    ' Radio City recently forayed into New Media in May 2008 with the launch of a music portal - PlanetRadiocity.com that offers music related news, videos, songs, and other music-related features.',
    ' The Radio station currently plays a mix of Hindi and Regional music.',
    ' Abraham Thomas is the CEO of the company.']]

In [5]:
test_hotpot_qa_json[0]

{'_id': '5a8b57f25542995d1e6f1371',
 'answer': 'yes',
 'question': 'Were Scott Derrickson and Ed Wood of the same nationality?',
 'supporting_facts': [['Scott Derrickson', 0], ['Ed Wood', 0]],
 'context': [['Adam Collis',
   ['Adam Collis is an American filmmaker and actor.',
    ' He attended the Duke University from 1986 to 1990 and the University of California, Los Angeles from 2007 to 2010.',
    ' He also studied cinema at the University of Southern California from 1991 to 1997.',
    ' Collis first work was the assistant director for the Scott Derrickson\'s short "Love in the Ruins" (1995).',
    ' In 1998, he played "Crankshaft" in Eric Koyanagi\'s "Hundred Percent".']],
  ['Ed Wood (film)',
   ['Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.',
    " The film concerns the period in Wood's life when he made his best-known films as well as his relationship with actor Bela Lug

In [6]:
for item in train_hotpot_qa_json[0]['context']:
    print(f"Title: {item[0]}")
    print(f"Paragraph: {item[1:]}")
    print("-" * 50)

Title: Radio City (Indian radio station)
Paragraph: [["Radio City is India's first private FM radio station and was started on 3 July 2001.", ' It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003).', ' It plays Hindi, English and regional songs.', ' It was launched in Hyderabad in March 2006, in Chennai on 7 July 2006 and in Visakhapatnam October 2007.', ' Radio City recently forayed into New Media in May 2008 with the launch of a music portal - PlanetRadiocity.com that offers music related news, videos, songs, and other music-related features.', ' The Radio station currently plays a mix of Hindi and Regional music.', ' Abraham Thomas is the CEO of the company.']]
--------------------------------------------------
Title: History of Albanian football
Paragraph: [['Football in Albania existed before the Albanian Football Federation (FSHF) was created.', " This was ev

In [7]:
def divide_and_batch(train_json, batch_size):
    batch_context = []
    batch_question = []
    batch_answer = []

    for i in range(0, len(train_json), batch_size):
        batch = train_json[i:i + batch_size]
        context = []
        question = []
        answer = []

        for item in batch:
            context_text = ""
            j = 1
            for ctx in item['context']:
                context_text += f"Title {j} : {ctx[0]} \nParagraph {j} : {''.join(ctx[1])}\n"
                j += 1
            context.append(context_text)
            question.append(item['question'])
            answer.append(item['answer'])

        batch_context.append(context)
        batch_question.append(question)
        batch_answer.append(answer)

    return batch_context, batch_question, batch_answer

In [8]:
if IS_DEBUG:
    train_hotpot_qa_json = train_hotpot_qa_json[:DEBUG_QUESTION_SIZE]
    test_hotpot_qa_json = test_hotpot_qa_json[:DEBUG_QUESTION_SIZE]
    batch_context, batch_question, batch_answer = divide_and_batch(train_hotpot_qa_json, BATCH_SIZE)
else:
    batch_context, batch_question, batch_answer = divide_and_batch(test_hotpot_qa_json, BATCH_SIZE)

In [9]:
for i in range(len(batch_context[0])):
    print(f"Batch {i + 1}:")
    print("Context:", batch_context[0][i])
    print("Question:", batch_question[0][i])
    print("Answer:", batch_answer[0][i])
    print("-" * 50)
    if i == 9:  # Limit to 3 batches for brevity
        break

Batch 1:
Context: Title 1 : Radio City (Indian radio station) 
Paragraph 1 : Radio City is India's first private FM radio station and was started on 3 July 2001. It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003). It plays Hindi, English and regional songs. It was launched in Hyderabad in March 2006, in Chennai on 7 July 2006 and in Visakhapatnam October 2007. Radio City recently forayed into New Media in May 2008 with the launch of a music portal - PlanetRadiocity.com that offers music related news, videos, songs, and other music-related features. The Radio station currently plays a mix of Hindi and Regional music. Abraham Thomas is the CEO of the company.
Title 2 : History of Albanian football 
Paragraph 2 : Football in Albania existed before the Albanian Football Federation (FSHF) was created. This was evidenced by the team's registration at the Balkan Cup tou

In [10]:
del train_hotpot_qa_json
del test_hotpot_qa_json

## 2. Getting Triples from Context and Question

In [11]:
from dotenv import load_dotenv
import os

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

In [12]:
from langchain_openai import ChatOpenAI
from langchain_core.globals import set_debug, set_verbose, set_llm_cache
from langchain_community.cache import InMemoryCache
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

set_debug(False)
set_verbose(False)
set_llm_cache(InMemoryCache())

llm_model = ChatOpenAI(model=LLM_MODEL, api_key=openai_api_key)

In [13]:
system_msg = """
<role>
    You are an expert in natural language understanding and knowledge graph construction. Your task is to extract structured relationship triples from a context set (Titles and Paragraphs), including both explicit and logically inferable facts.
</role>

<behavior>
    <rule name="Entity Extraction">
        Identify and extract unique entities from Titles and Paragraphs. Use canonical forms for names. Capture entities including people, organizations, locations, publications, dates, and events.
    </rule>
    <rule name="Relationship Extraction">
        Extract explicit and clearly implied relationships between entities in the form of (subject, relation, object) triples.
        This includes:
        - Temporal relations such as publication or operational time spans (e.g., "published during", "active from", "merged in").
        - Authorship, editorial, or contribution roles (e.g., "edited by", "featured work by").
        - Event-based or structural relations (e.g., "merged into", "founded by", "took over").
    </rule>
    <rule name="Completeness">
        Extract **as many valid and informative triples as possible**. Include relationships that are inferred logically but unambiguous in context (e.g., date ranges, geographic associations, contributor roles).
    </rule>
    <rule name="Formatting and Output">
        Output must contain two sections, in order:
        1. ### CONTEXT_REASONING — step-by-step reasoning describing how entities and relationships were selected or inferred.
        2. ### CONTEXT_TRIPLES — list all extracted triples, each on its own line in the format (entity_1, relationship, entity_2)
        Do not add explanations, numbering, or extra text outside of these sections.
    </rule>
</behavior>

<format>
1. Carefully read the <context> section.
2. Begin with this heading:
   ### CONTEXT_REASONING
   Then, write concise reasoning steps describing how entities and relationships were selected and interpreted.
3. After reasoning, print this heading:
   ### CONTEXT_TRIPLES
   Then list each extracted triple on a new line in the format (entity_1, relationship, entity_2).
4. Do not include any explanations, text, or comments after the last triple.
</format>
"""

human_msg = """
<context>
{context}
</context>
"""

prompt = ChatPromptTemplate([("system", system_msg), ("human", human_msg)])

In [14]:
chain = prompt | llm_model | StrOutputParser()

In [15]:
batch_input_list = []
main_df = pd.DataFrame()

for i in range(len(batch_context)):
    batch_num_list = [i] * len(batch_context[i])
    question_num_list = list(range(1, len(batch_context[i]) + 1))
    context_list = []
    question_list = []
    answers_list = []

    batch_input_item = []
    j = 0
    for batch in range(len(batch_context[i])):
        context = batch_context[i][batch]
        batch_input_item.append({
            "context": context
        })
        context_list.append(context)
        question_list.append(batch_question[i][j])
        answers_list.append(batch_answer[i][j])
        j += 1
    batch_input_list.append(batch_input_item)
    df = pd.DataFrame({
        "batch_num": batch_num_list,
        "question_num": question_num_list,
        "context": context_list,
        "question": question_list,
        "answer": answers_list
    })
    main_df = pd.concat([main_df, df], ignore_index=True)

for i, batch in enumerate(batch_input_list):
    print(f"Batch {i + 1}:")
    for item in batch:
        print("Context:", item['context'])
        print("-" * 50)

Batch 1:
Context: Title 1 : Radio City (Indian radio station) 
Paragraph 1 : Radio City is India's first private FM radio station and was started on 3 July 2001. It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003). It plays Hindi, English and regional songs. It was launched in Hyderabad in March 2006, in Chennai on 7 July 2006 and in Visakhapatnam October 2007. Radio City recently forayed into New Media in May 2008 with the launch of a music portal - PlanetRadiocity.com that offers music related news, videos, songs, and other music-related features. The Radio station currently plays a mix of Hindi and Regional music. Abraham Thomas is the CEO of the company.
Title 2 : History of Albanian football 
Paragraph 2 : Football in Albania existed before the Albanian Football Federation (FSHF) was created. This was evidenced by the team's registration at the Balkan Cup tou

In [16]:
main_df

,batch_num,question_num,context,question,answer
0,0,1,Title 1 : Radio City (Indian radio station) \n...,Which magazine was started first Arthur's Maga...,Arthur's Magazine
1,0,2,Title 1 : Ritz-Carlton Jakarta \nParagraph 1 :...,The Oberoi family is part of a hotel company t...,Delhi
2,0,3,Title 1 : Lisa Simpson \nParagraph 1 : Lisa Ma...,Musician and satirist Allie Goertz wrote a son...,President Richard Nixon
3,0,4,"Title 1 : Moloch: or, This Gentile World \nPar...",What nationality was James Henry Miller's wife?,American
4,0,5,Title 1 : Cadmium chloride \nParagraph 1 : Cad...,Cadmium Chloride is slightly soluble in this c...,alcohol
5,0,6,Title 1 : Li Na \nParagraph 1 : Li Na (; ; bor...,Which tennis player won more Grand Slam titles...,Jonathan Stark
6,0,7,"Title 1 : India \nParagraph 1 : India, officia...",Which genus of moth in the world's seventh-lar...,Crambidae
7,0,8,Title 1 : Verano de Escándalo (1998) \nParagra...,Who was once considered the best kick boxer in...,Badr Hari
8,0,9,Title 1 : House of Anubis \nParagraph 1 : Hous...,"The Dutch-Belgian television series that ""Hous...",2006
9,0,10,Title 1 : Mount Panorama Circuit \nParagraph 1...,What is the length of the track where the 2013...,6.213 km long


In [17]:
graphs_str_batch = []

for i, batch in enumerate(batch_input_list):
    print(f"Batch {i + 1} ... processing {len(batch)} items")
    graphs_str_list = chain.batch(batch)
    graphs_str_batch.append(graphs_str_list)

Batch 1 ... processing 10 items


In [18]:
for i, graphs_str_list in enumerate(graphs_str_batch):
    print(f"Batch {i + 1} ... processed {len(graphs_str_list)} items")
    for j, graph_str in enumerate(graphs_str_list):
        print(f"Item {j + 1}:")
        print(graph_str)
        print("-" * 50)

Batch 1 ... processed 10 items
Item 1:
### CONTEXT_REASONING
Entities were identified by reviewing titles and paragraphs for unique names of organizations, people, places, dates, publications, and events. For relationship extraction, I selected explicit relationships (like "founded by", "edited by", "launched in", "published by", "certified by", etc.) and inferred logical relationships (like geographic locations, temporal spans, participation in events, evolution of organizations, etc.) where information was clear and unambiguous. Temporal spans (start/end dates), mergers, certifications, and transitions in organization structures were systematically extracted. Relationships involving products (e.g., songs, albums, clothing lines) and their associations (e.g., certified by, released on) were included when directly mentioned or unequivocally implied. 

### CONTEXT_TRIPLES
(Radio City, is a, FM radio station)
(Radio City, is located in, India)
(Radio City, was started on, 3 July 2001)
(R

In [19]:
graphs_str_df = pd.DataFrame({
    "batch_num": [],
    "question_num": [],
    "graphs_str": []
})

for i, graphs_str_list in enumerate(graphs_str_batch):
    batch_num_list = [i] * len(graphs_str_list)
    question_num_list = list(range(1, len(graphs_str_list) + 1))
    graphs_str_df = pd.concat([graphs_str_df, pd.DataFrame({
        "batch_num": batch_num_list,
        "question_num": question_num_list,
        "graphs_str": graphs_str_list
    })], ignore_index=True)

del graphs_str_batch
del graphs_str_list
del batch_input_list
del batch_context
del batch_question
del batch_answer

graphs_str_df

,batch_num,question_num,graphs_str
0,0.0,1.0,### CONTEXT_REASONING\nEntities were identifie...
1,0.0,2.0,"### CONTEXT_REASONING\nFirst, I reviewed each ..."
2,0.0,3.0,### CONTEXT_REASONING\nEntities selected inclu...
3,0.0,4.0,### CONTEXT_REASONING\nI first identified uniq...
4,0.0,5.0,"### CONTEXT_REASONING\nFirst, I identified ent..."
5,0.0,6.0,### CONTEXT_REASONING\nEntities were extracted...
6,0.0,7.0,### CONTEXT_REASONING\nEntities were identifie...
7,0.0,8.0,### CONTEXT_REASONING\nI read through each tit...
8,0.0,9.0,### CONTEXT_REASONING\nEntities were extracted...
9,0.0,10.0,### CONTEXT_REASONING\nEntities were identifie...


## 3. Parsing the Output and Building KGs

In [20]:
import re
from ast import literal_eval

def sanitize_str(name):
    """
    Sanitize a name by removing extra spaces, apostrophes, and ensuring proper formatting.
    """
    # Replace apostrophes with empty string
    name = name.replace("'s", "s")
    name = name.replace("'", "")

    # Replace spaces with underscores
    name = name.replace(" ", "_")
    name = name.replace("–", "_")

    # Remove any other problematic characters
    name = re.sub(r'[^\w\-_]', '', name)

    # Prefix if starts with digit
    if re.match(r"^\d", name):
        name = f"n{name}"

    # If empty string, return something safe
    if not name:
        name = "unknown"

    return name

def sanitize_entities_and_relations(triple):
    """
    Sanitize entities and relations in a triple by removing extra spaces and ensuring proper formatting.
    """
    return (
        sanitize_str(triple[0].strip().lower()),  # Subject
        sanitize_str(triple[1].strip().lower()),  # Relation
        sanitize_str(triple[2].strip().lower())   # Object
    )


def parse_custom_triples(triples_str):
    triples = []
    for line in triples_str.strip().splitlines():
        line = line.strip()
        if not line:
            continue
        # Remove enclosing parentheses
        if line.startswith('(') and line.endswith(')'):
            line = line[1:-1]
        # Now, split ONLY on the first two commas
        parts = []
        remaining = line
        for _ in range(2):
            # Find the first comma
            idx = remaining.find(',')
            if idx == -1:
                break
            parts.append(remaining[:idx].strip())
            remaining = remaining[idx+1:].strip()
        parts.append(remaining)
        if len(parts) == 3:
            triples.append(sanitize_entities_and_relations(tuple(parts)))
    return triples

def extract_sections(text, reasoning_pat, triples_pat):
    # Extract sections using regex
    extracted_reasoning = re.search(reasoning_pat, text, re.DOTALL)
    extracted_triples = re.search(triples_pat, text, re.DOTALL)

    # Clean reasoning sections
    reasoning_str = extracted_reasoning.group(1).strip() if extracted_reasoning else ""

    # Use the custom parser for triples
    triples_list = parse_custom_triples(extracted_triples.group(1)) if extracted_triples else []

    return reasoning_str, triples_list

def extract_context_to_columns(row):
    # Regex patterns for section headers
    context_reasoning_pat = r'### CONTEXT_REASONING\s*(.*?)\s*### CONTEXT_TRIPLES'
    context_triples_pat = r'### CONTEXT_TRIPLES\s*(.*)'

    context_reasoning, context_triples = extract_sections(row['graphs_str'], context_reasoning_pat, context_triples_pat)
    return pd.Series([context_reasoning, context_triples],
                     index=['context_reasoning', 'context_triples'])

In [21]:
graphs_str_df[['context_reasoning', 'context_triples']] = graphs_str_df.apply(extract_context_to_columns, axis=1)
graphs_str_df

,batch_num,question_num,graphs_str,context_reasoning,context_triples
0,0.0,1.0,### CONTEXT_REASONING\nEntities were identifie...,Entities were identified by reviewing titles a...,"[(radio_city, is_a, fm_radio_station), (radio_..."
1,0.0,2.0,"### CONTEXT_REASONING\nFirst, I reviewed each ...","First, I reviewed each paragraph and title for...","[(ritz-carlton_jakarta, is_located_in, jakarta..."
2,0.0,3.0,### CONTEXT_REASONING\nEntities selected inclu...,Entities selected include canonical names for ...,"[(lisa_simpson, is_character_in, the_simpsons)..."
3,0.0,4.0,### CONTEXT_REASONING\nI first identified uniq...,"I first identified unique entities, focusing o...","[(moloch_or, this_gentile_world, is_a_novel), ..."
4,0.0,5.0,"### CONTEXT_REASONING\nFirst, I identified ent...","First, I identified entities such as compounds...","[(cadmium_chloride, has_formula, cdcl), (cadmi..."
5,0.0,6.0,### CONTEXT_REASONING\nEntities were extracted...,Entities were extracted based on person names ...,"[(li_na, born_on, n26_february_1982), (li_na, ..."
6,0.0,7.0,### CONTEXT_REASONING\nEntities were identifie...,"Entities were identified as countries (India, ...","[(india, official_name, republic_of_india), (i..."
7,0.0,8.0,### CONTEXT_REASONING\nI read through each tit...,"I read through each title and paragraph, ident...","[(verano_de_escándalo_1998, is_a, professional..."
8,0.0,9.0,### CONTEXT_REASONING\nEntities were extracted...,"Entities were extracted from series titles, cr...","[(house_of_anubis, is_a_television_series_deve..."
9,0.0,10.0,### CONTEXT_REASONING\nEntities were identifie...,Entities were identified by parsing titles and...,"[(mount_panorama_circuit, located_in, bathurst..."


In [22]:
main_df = main_df.merge(graphs_str_df, on=['batch_num', 'question_num'], how='left')
# del graphs_str_df
main_df

,batch_num,question_num,context,question,answer,graphs_str,context_reasoning,context_triples
0,0,1,Title 1 : Radio City (Indian radio station) \n...,Which magazine was started first Arthur's Maga...,Arthur's Magazine,### CONTEXT_REASONING\nEntities were identifie...,Entities were identified by reviewing titles a...,"[(radio_city, is_a, fm_radio_station), (radio_..."
1,0,2,Title 1 : Ritz-Carlton Jakarta \nParagraph 1 :...,The Oberoi family is part of a hotel company t...,Delhi,"### CONTEXT_REASONING\nFirst, I reviewed each ...","First, I reviewed each paragraph and title for...","[(ritz-carlton_jakarta, is_located_in, jakarta..."
2,0,3,Title 1 : Lisa Simpson \nParagraph 1 : Lisa Ma...,Musician and satirist Allie Goertz wrote a son...,President Richard Nixon,### CONTEXT_REASONING\nEntities selected inclu...,Entities selected include canonical names for ...,"[(lisa_simpson, is_character_in, the_simpsons)..."
3,0,4,"Title 1 : Moloch: or, This Gentile World \nPar...",What nationality was James Henry Miller's wife?,American,### CONTEXT_REASONING\nI first identified uniq...,"I first identified unique entities, focusing o...","[(moloch_or, this_gentile_world, is_a_novel), ..."
4,0,5,Title 1 : Cadmium chloride \nParagraph 1 : Cad...,Cadmium Chloride is slightly soluble in this c...,alcohol,"### CONTEXT_REASONING\nFirst, I identified ent...","First, I identified entities such as compounds...","[(cadmium_chloride, has_formula, cdcl), (cadmi..."
5,0,6,Title 1 : Li Na \nParagraph 1 : Li Na (; ; bor...,Which tennis player won more Grand Slam titles...,Jonathan Stark,### CONTEXT_REASONING\nEntities were extracted...,Entities were extracted based on person names ...,"[(li_na, born_on, n26_february_1982), (li_na, ..."
6,0,7,"Title 1 : India \nParagraph 1 : India, officia...",Which genus of moth in the world's seventh-lar...,Crambidae,### CONTEXT_REASONING\nEntities were identifie...,"Entities were identified as countries (India, ...","[(india, official_name, republic_of_india), (i..."
7,0,8,Title 1 : Verano de Escándalo (1998) \nParagra...,Who was once considered the best kick boxer in...,Badr Hari,### CONTEXT_REASONING\nI read through each tit...,"I read through each title and paragraph, ident...","[(verano_de_escándalo_1998, is_a, professional..."
8,0,9,Title 1 : House of Anubis \nParagraph 1 : Hous...,"The Dutch-Belgian television series that ""Hous...",2006,### CONTEXT_REASONING\nEntities were extracted...,"Entities were extracted from series titles, cr...","[(house_of_anubis, is_a_television_series_deve..."
9,0,10,Title 1 : Mount Panorama Circuit \nParagraph 1...,What is the length of the track where the 2013...,6.213 km long,### CONTEXT_REASONING\nEntities were identifie...,Entities were identified by parsing titles and...,"[(mount_panorama_circuit, located_in, bathurst..."


In [23]:
# Loop through each row. Save the context triples as OWL KG files
from rdflib import Graph, Namespace, RDF, RDFS, OWL, URIRef
import os

def save_kg_from_triples(triples, batch_num, question_num):
    g = Graph()
    EX = Namespace("http://example.org/")

    # Add triples to the graph
    for subject, relation, obj in triples:
        subject_uri = URIRef(EX[subject])
        object_uri = URIRef(EX[obj])
        g.add((subject_uri, URIRef(EX[relation]), object_uri))

    # Define the file path
    file_path = f"{KNOWLEDGE_GRAPH_PATH}/{DATASET_NAME}_ontology_b{batch_num}_q{question_num}.rdf"

    # Save the graph in OWL format
    g.serialize(destination=file_path, format='xml')
    print(f"Saved KG for batch {batch_num}, question {question_num} to {file_path}")

for index, row in main_df.iterrows():
    batch_num = row['batch_num']
    question_num = row['question_num']
    context_triples = row['context_triples']

    if context_triples:  # Only save if there are context triples
        save_kg_from_triples(context_triples, batch_num, question_num)

Saved KG for batch 0, question 1 to outputs/knowledge_graphs/hotpotQA_ontology_b0_q1.rdf
Saved KG for batch 0, question 2 to outputs/knowledge_graphs/hotpotQA_ontology_b0_q2.rdf
Saved KG for batch 0, question 3 to outputs/knowledge_graphs/hotpotQA_ontology_b0_q3.rdf
Saved KG for batch 0, question 4 to outputs/knowledge_graphs/hotpotQA_ontology_b0_q4.rdf
Saved KG for batch 0, question 5 to outputs/knowledge_graphs/hotpotQA_ontology_b0_q5.rdf
Saved KG for batch 0, question 6 to outputs/knowledge_graphs/hotpotQA_ontology_b0_q6.rdf
Saved KG for batch 0, question 7 to outputs/knowledge_graphs/hotpotQA_ontology_b0_q7.rdf
Saved KG for batch 0, question 8 to outputs/knowledge_graphs/hotpotQA_ontology_b0_q8.rdf
Saved KG for batch 0, question 9 to outputs/knowledge_graphs/hotpotQA_ontology_b0_q9.rdf
Saved KG for batch 0, question 10 to outputs/knowledge_graphs/hotpotQA_ontology_b0_q10.rdf


## 4. Extracting Entities from Questions

In [24]:
batch_input_list = []

for i in range(0, main_df.shape[0], BATCH_SIZE):
    batch_df = main_df.iloc[i:i+BATCH_SIZE]
    batch_input_item = []

    for i, (index, row) in enumerate(batch_df.iterrows()):
        batch_input_item.append({
            "question": row['question']
        })
    batch_input_list.append(batch_input_item)

for i, batch in enumerate(batch_input_list):
    print(f"Batch {i + 1}:")
    for item in batch:
        print("Question:", item['question'])
        print("-" * 50)

Batch 1:
Question: Which magazine was started first Arthur's Magazine or First for Women?
--------------------------------------------------
Question: The Oberoi family is part of a hotel company that has a head office in what city?
--------------------------------------------------
Question: Musician and satirist Allie Goertz wrote a song about the "The Simpsons" character Milhouse, who Matt Groening named after who?
--------------------------------------------------
Question:  What nationality was James Henry Miller's wife?
--------------------------------------------------
Question: Cadmium Chloride is slightly soluble in this chemical, it is also called what?
--------------------------------------------------
Question: Which tennis player won more Grand Slam titles, Henri Leconte or Jonathan Stark?
--------------------------------------------------
Question: Which genus of moth in the world's seventh-largest country contains only one species?
---------------------------------------

In [25]:
system_msg = """
<role>
    You are an expert in natural language understanding and knowledge graph construction. Your task is to convert a provided question into a set of question triples, using simple variable names (such as x, y, z, i, j, k) for unknown entities or values to be inferred.
</role>

<behavior>
    <rule name="Entity Extraction">
        Identify and extract unique entities. Use canonical names.
    </rule>
    <rule name="Relationship Extraction">
        Extract explicit or clear implicit relationships between entities as (subject, relation, object) triples.
    </rule>
    <rule name="Temporal and Comparative Relations">
        Use relations like "started in", "founded by", "wrote a song about", "is named after", etc., matching the logic and semantics in the context or question.
    </rule>
    <rule name="Variable Placeholders for Unknowns">
        For each unknown or answer to be found in the question, use a simple English variable (x, y, z, i, j, k, etc.) in place of the answer within the triples. Each distinct unknown in the question gets a unique variable.
    </rule>
    <rule name="Question Triple Decomposition">
        For each question, decompose it into one or more triples, using variable placeholders for any unknowns to be inferred from the context.
        Examples:
        - "Which magazine was started first Arthur's Magazine or First for Women?"
           (Arthur's Magazine, started in, x)
           (First for Women, started in, y)
        - "The Oberoi family is part of a hotel company that has a head office in what city?"
           (Oberoi family, is part of, x)
           (x, has head office in, y)
    </rule>
    <rule name="Formatting and Output">
        Output must contain four sections, in order:
        1. ### QUESTION_REASONING — a step-by-step reasoning of how the question was decomposed into question triples
        2. ### QUESTIONS_TRIPLES — list the question triples, each on its own line. each on its own line in the format (entity_1, relationship, entity_2)
        Do not add explanations, numbering, or extra text.
    </rule>
</behavior>

<format>
1. Carefully read the <question> section.
2. Before extracting question triples, explain step by step how you break down the question and assign variable placeholders. Print this heading:
   ### QUESTION_REASONING
   Then write your reasoning in clear, concise steps.
3. After reasoning, print the question triples. Print this heading:
   ### QUESTIONS_TRIPLES
   List each question triple on a new line, using variable placeholders (x, y, z, etc.) for unknowns.
4. After the last question triple, end with no extra text or parentheses.
</format>
"""

human_msg = """
<question>
{question}
</question>
"""

prompt = ChatPromptTemplate([("system", system_msg), ("human", human_msg)])

In [26]:
chain = prompt | llm_model | StrOutputParser()

In [27]:
question_str_batch = []

for i, batch in enumerate(batch_input_list):
    print(f"Batch {i + 1} ... processing {len(batch)} items")
    question_str_list = chain.batch(batch)
    question_str_batch.append(question_str_list)

Batch 1 ... processing 10 items


In [28]:
for i, question_str_list in enumerate(question_str_batch):
    print(f"Batch {i + 1} ... processed {len(question_str_list)} items")
    for j, graph_str in enumerate(question_str_list):
        print(f"Item {j + 1}:")
        print(graph_str)
        print("-" * 50)

Batch 1 ... processed 10 items
Item 1:
### QUESTION_REASONING
The question is asking for a comparison between two magazines: Arthur's Magazine and First for Women. The explicit property in question is the start date ("was started first"), which involves the dates each magazine was started. Since the actual start dates are unknowns, assign variables (x for Arthur's Magazine and y for First for Women). To infer which was "started first," we need both start years as separate unknowns in the triples.

### QUESTIONS_TRIPLES
(Arthur's Magazine, started in, x)
(First for Women, started in, y)
--------------------------------------------------
Item 2:
### QUESTION_REASONING
First, I identify the main entities mentioned: "Oberoi family", "hotel company", and "city" (unknown). The question asks for the city where the hotel company (of which the Oberoi family is part) has its head office. The specific hotel company is not named but is associated with the Oberoi family. The unknown to find is the 

In [29]:
questions_str_df = pd.DataFrame({
    "batch_num": [],
    "question_num": [],
    "questions_str": []
})

for i, graphs_str_list in enumerate(question_str_batch):
    batch_num_list = [i] * len(graphs_str_list)
    question_num_list = list(range(1, len(graphs_str_list) + 1))
    questions_str_df = pd.concat([questions_str_df, pd.DataFrame({
        "batch_num": batch_num_list,
        "question_num": question_num_list,
        "questions_str": graphs_str_list
    })], ignore_index=True)

questions_str_df

,batch_num,question_num,questions_str
0,0.0,1.0,### QUESTION_REASONING\nThe question is asking...
1,0.0,2.0,"### QUESTION_REASONING\nFirst, I identify the ..."
2,0.0,3.0,"### QUESTION_REASONING\nFirst, identify the en..."
3,0.0,4.0,### QUESTION_REASONING\nThe question is asking...
4,0.0,5.0,"### QUESTION_REASONING\nFirst, identify the st..."
5,0.0,6.0,### QUESTION_REASONING\nThe question is compar...
6,0.0,7.0,### QUESTION_REASONING\n1. The question is ask...
7,0.0,8.0,### QUESTION_REASONING\nThe question is asking...
8,0.0,9.0,### QUESTION_REASONING\n1. Identify the main e...
9,0.0,10.0,### QUESTION_REASONING\n1. Identify the main e...


In [30]:
def extract_question_to_columns(row):
    # Regex patterns for section headers
    questions_reasoning_pat = r'### QUESTION_REASONING\s*(.*?)\s*### QUESTIONS_TRIPLES'
    questions_triples_pat = r'### QUESTIONS_TRIPLES\s*(.*)'

    questions_reasoning, questions_triples = extract_sections(row['questions_str'], questions_reasoning_pat, questions_triples_pat)
    return pd.Series([questions_reasoning, questions_triples],
                     index=['question_reasoning', 'question_triples'])

questions_str_df[['question_reasoning', 'question_triples']] = questions_str_df.apply(extract_question_to_columns, axis=1)
questions_str_df

,batch_num,question_num,questions_str,question_reasoning,question_triples
0,0.0,1.0,### QUESTION_REASONING\nThe question is asking...,The question is asking for a comparison betwee...,"[(arthurs_magazine, started_in, x), (first_for..."
1,0.0,2.0,"### QUESTION_REASONING\nFirst, I identify the ...","First, I identify the main entities mentioned:...","[(oberoi_family, is_part_of, y), (y, has_head_..."
2,0.0,3.0,"### QUESTION_REASONING\nFirst, identify the en...","First, identify the entities: Allie Goertz, ""T...","[(allie_goertz, wrote_a_song_about, milhouse),..."
3,0.0,4.0,### QUESTION_REASONING\nThe question is asking...,The question is asking about the nationality o...,"[(james_henry_miller, has_wife, x), (x, has_na..."
4,0.0,5.0,"### QUESTION_REASONING\nFirst, identify the st...","First, identify the statements within the ques...","[(cadmium_chloride, is_slightly_soluble_in, x)..."
5,0.0,6.0,### QUESTION_REASONING\nThe question is compar...,The question is comparing the number of Grand ...,"[(henri_leconte, won_grand_slam_titles, x), (j..."
6,0.0,7.0,### QUESTION_REASONING\n1. The question is ask...,1. The question is asking for the genus of mot...,"[(x, is_a_genus_of_moth_in, y), (y, is_the_wor..."
7,0.0,8.0,### QUESTION_REASONING\nThe question is asking...,The question is asking for the identity of a p...,"[(x, was_once_considered, best_kick_boxer_in_t..."
8,0.0,9.0,### QUESTION_REASONING\n1. Identify the main e...,1. Identify the main entities in the question:...,"[(house_of_anubis, was_based_on, x), (x, first..."
9,0.0,10.0,### QUESTION_REASONING\n1. Identify the main e...,1. Identify the main entities in the question:...,"[(n2013_liqui_moly_bathurst_12_hour, was_stage..."


In [31]:
main_df = main_df.merge(questions_str_df, on=['batch_num', 'question_num'], how='left')
# del graphs_str_df
main_df

,batch_num,question_num,context,question,answer,graphs_str,context_reasoning,context_triples,questions_str,question_reasoning,question_triples
0,0,1,Title 1 : Radio City (Indian radio station) \n...,Which magazine was started first Arthur's Maga...,Arthur's Magazine,### CONTEXT_REASONING\nEntities were identifie...,Entities were identified by reviewing titles a...,"[(radio_city, is_a, fm_radio_station), (radio_...",### QUESTION_REASONING\nThe question is asking...,The question is asking for a comparison betwee...,"[(arthurs_magazine, started_in, x), (first_for..."
1,0,2,Title 1 : Ritz-Carlton Jakarta \nParagraph 1 :...,The Oberoi family is part of a hotel company t...,Delhi,"### CONTEXT_REASONING\nFirst, I reviewed each ...","First, I reviewed each paragraph and title for...","[(ritz-carlton_jakarta, is_located_in, jakarta...","### QUESTION_REASONING\nFirst, I identify the ...","First, I identify the main entities mentioned:...","[(oberoi_family, is_part_of, y), (y, has_head_..."
2,0,3,Title 1 : Lisa Simpson \nParagraph 1 : Lisa Ma...,Musician and satirist Allie Goertz wrote a son...,President Richard Nixon,### CONTEXT_REASONING\nEntities selected inclu...,Entities selected include canonical names for ...,"[(lisa_simpson, is_character_in, the_simpsons)...","### QUESTION_REASONING\nFirst, identify the en...","First, identify the entities: Allie Goertz, ""T...","[(allie_goertz, wrote_a_song_about, milhouse),..."
3,0,4,"Title 1 : Moloch: or, This Gentile World \nPar...",What nationality was James Henry Miller's wife?,American,### CONTEXT_REASONING\nI first identified uniq...,"I first identified unique entities, focusing o...","[(moloch_or, this_gentile_world, is_a_novel), ...",### QUESTION_REASONING\nThe question is asking...,The question is asking about the nationality o...,"[(james_henry_miller, has_wife, x), (x, has_na..."
4,0,5,Title 1 : Cadmium chloride \nParagraph 1 : Cad...,Cadmium Chloride is slightly soluble in this c...,alcohol,"### CONTEXT_REASONING\nFirst, I identified ent...","First, I identified entities such as compounds...","[(cadmium_chloride, has_formula, cdcl), (cadmi...","### QUESTION_REASONING\nFirst, identify the st...","First, identify the statements within the ques...","[(cadmium_chloride, is_slightly_soluble_in, x)..."
5,0,6,Title 1 : Li Na \nParagraph 1 : Li Na (; ; bor...,Which tennis player won more Grand Slam titles...,Jonathan Stark,### CONTEXT_REASONING\nEntities were extracted...,Entities were extracted based on person names ...,"[(li_na, born_on, n26_february_1982), (li_na, ...",### QUESTION_REASONING\nThe question is compar...,The question is comparing the number of Grand ...,"[(henri_leconte, won_grand_slam_titles, x), (j..."
6,0,7,"Title 1 : India \nParagraph 1 : India, officia...",Which genus of moth in the world's seventh-lar...,Crambidae,### CONTEXT_REASONING\nEntities were identifie...,"Entities were identified as countries (India, ...","[(india, official_name, republic_of_india), (i...",### QUESTION_REASONING\n1. The question is ask...,1. The question is asking for the genus of mot...,"[(x, is_a_genus_of_moth_in, y), (y, is_the_wor..."
7,0,8,Title 1 : Verano de Escándalo (1998) \nParagra...,Who was once considered the best kick boxer in...,Badr Hari,### CONTEXT_REASONING\nI read through each tit...,"I read through each title and paragraph, ident...","[(verano_de_escándalo_1998, is_a, professional...",### QUESTION_REASONING\nThe question is asking...,The question is asking for the identity of a p...,"[(x, was_once_considered, best_kick_boxer_in_t..."
8,0,9,Title 1 : House of Anubis \nParagraph 1 : Hous...,"The Dutch-Belgian television series that ""Hous...",2006,### CONTEXT_REASONING\nEntities were extracted...,"Entities were extracted from series titles, cr...","[(house_of_anubis, is_a_television_series_deve...",### QUESTION_REASONING\n1. Identify the main e...,1. Identify the main entities in the question:...,"[(house_of_anubis, was_based_on, x), (x, first..."
9,0,10,Title 1 : Mount Panor

## 5. Extracting sub-graphs from KGs

In [32]:
import threading
import numpy as np
from queue import Queue
from langchain_openai import OpenAIEmbeddings
import ast

def short_name(uri):
    s = str(uri)
    if '#' in s:
        return s.split('#')[-1]
    elif '/' in s:
        return s.split('/')[-1]
    return s

def load_embedding_dict(csv_path):
    embedding_dict = {}
    if os.path.exists(csv_path):
        print(f"Loading existing embeddings from '{csv_path}'...")
        df = pd.read_csv(csv_path)
        for _, row in df.iterrows():
            emb = ast.literal_eval(row["embedding"]) if isinstance(row["embedding"], str) else row["embedding"]
            embedding_dict[row["string"]] = emb
    else:
        print(f"No embedding cache found. Initializing '{csv_path}' as empty.")
        pd.DataFrame(columns=["string", "embedding"]).to_csv(csv_path, index=False)
    return embedding_dict

def get_embedding(text, embedding_dict, lock, embedder):
    with lock:
        if text in embedding_dict:
            print(f"Embedding for '{text}' found in cache.")
            return embedding_dict[text]
    print(f"Requesting embedding for '{text}'...")
    emb = embedder.embed_query(text)
    with lock:
        embedding_dict[text] = emb
    print(f"Embedding for '{text}' cached.")
    return emb

def cosine_similarity(vec1, vec2):
    v1 = np.array(vec1)
    v2 = np.array(vec2)
    if np.linalg.norm(v1) == 0 or np.linalg.norm(v2) == 0:
        return 0.0
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

def bfs_n_hop_triples(graph, start_entity, n):
    visited = set([start_entity])
    current_level = set([start_entity])
    neighborhood_triples = set()
    for hop in range(n):
        next_level = set()
        for node in current_level:
            # Outgoing
            for s, p, o in graph.triples((node, None, None)):
                next_level.add(o)
                neighborhood_triples.add((short_name(s), short_name(p), short_name(o)))
            # Incoming
            for s, p, o in graph.triples((None, None, node)):
                next_level.add(s)
                neighborhood_triples.add((short_name(s), short_name(p), short_name(o)))
        next_level -= visited
        if not next_level:
            break
        visited |= next_level
        current_level = next_level
    return list(neighborhood_triples)

def extract_entities(graph):
    entities = set()
    for s, p, o in graph:
        entities.add(s)
        entities.add(o)
    return list(entities)

def worker(entity_queue, graph, key_embedding, per, n, embedding_dict, lock, neighborhoods, matched_entities, embedder, thread_id):
    print(f"[Thread {thread_id}] started.")
    while not entity_queue.empty():
        entity = entity_queue.get()
        try:
            entity_label = short_name(entity)
            emb = get_embedding(entity_label, embedding_dict, lock, embedder)
            sim = cosine_similarity(key_embedding, emb)
            if sim >= per:
                triples = bfs_n_hop_triples(graph, entity, n)
                with lock:
                    neighborhoods.extend(triples)
                    matched_entities.append(entity_label)
        except Exception as e:
            print(f"[Thread {thread_id}] Error processing entity {entity}: {e}")
        finally:
            entity_queue.task_done()
    print(f"[Thread {thread_id}] finished.")

def kg_neighborhood_extractor(
    rdf_path, per, n, key, k=4, embedding_model="text-embedding-3-small", embedding_csv_path="embedding_dict.csv"
):
    print("Loading RDF graph...")
    graph = Graph()
    graph.parse(rdf_path)
    print("Extracting entities...")
    entities = extract_entities(graph)
    print(f"Total entities found: {len(entities)}")

    embedding_dict = load_embedding_dict(embedding_csv_path)
    neighborhoods = []
    matched_entities = []
    lock = threading.Lock()

    print(f"Initializing LangChain OpenAIEmbeddings with model '{embedding_model}'...")
    embedder = OpenAIEmbeddings(model=embedding_model)

    print(f"Getting embedding for search key: '{key}'")
    key_embedding = get_embedding(key, embedding_dict, lock, embedder)

    entity_queue = Queue()
    for entity in entities:
        entity_queue.put(entity)

    threads = []
    for i in range(k):
        t = threading.Thread(
            target=worker,
            args=(
                entity_queue,
                graph,
                key_embedding,
                per,
                n,
                embedding_dict,
                lock,
                neighborhoods,
                matched_entities,
                embedder,
                i+1,
            ),
        )
        t.start()
        threads.append(t)

    entity_queue.join()
    for t in threads:
        t.join()

    # Deduplicate triples
    neighborhoods = list({triple for triple in neighborhoods})

    # Save embedding_dict to CSV
    with lock:
        rows = [
            {"string": s, "embedding": ",".join(map(str, v))}
            for s, v in embedding_dict.items()
        ]
        df = pd.DataFrame(rows)
        df.to_csv("embedding_dict.csv", index=False)

    print("\nSummary:")
    print(f"Matched entities: {matched_entities}")
    print(f"Total unique triples in neighborhoods: {len(neighborhoods)}")
    print("Embeddings saved as 'embedding_dict.csv'")
    return neighborhoods

In [33]:
# neighborhoods = kg_neighborhood_extractor(
#     rdf_path="outputs/knowledge_graphs/hotpotQA_ontology_b0_q1.rdf",
#     per=0.75, n=2, key="arthurs_magazine", k=3, embedding_model="text-embedding-3-small"
# )
# print("\nNeighborhood triples:")
# for triple in neighborhoods:
#     print(triple)

Loading RDF graph...
Extracting entities...
Total entities found: 148
Loading existing embeddings from 'embedding_dict.csv'...
Initializing LangChain OpenAIEmbeddings with model 'text-embedding-3-small'...
Getting embedding for search key: 'arthurs_magazine'
Embedding for 'arthurs_magazine' found in cache.
[Thread 1] started.
Embedding for 'arthurs_magazine' found in cache.
Embedding for 'bengaluru' found in cache.
Requesting embedding for 'a_day_later'...
[Thread 2] started.
Requesting embedding for 'n2014_year-end'...
[Thread 3] started.
Embedding for 'may_1846' found in cache.
Embedding for 'echosmith' found in cache.
Requesting embedding for 'first_college_established_specifically_for_women_in_south'...
Embedding for 'n2014_year-end' cached.
Embedding for 'bauer_media_group' found in cache.
Requesting embedding for 'perhaps_the_smallest_courthouse_in_united_states'...
Embedding for 'first_college_established_specifically_for_women_in_south' cached.
Requesting embedding for 'music_v

In [41]:
# loop through each row in main_df and extract neighborhoods
batch_input_list = []

for i in range(0, main_df.shape[0], BATCH_SIZE):
    batch_df = main_df.iloc[i:i+BATCH_SIZE]
    batch_input_item = []

    for i, (index, row) in enumerate(batch_df.iterrows()):
        reference_triples = []
        for triple in row['question_triples']:
            # Extract reference_triples using question_triples
            if len(triple[0]) > 1:
                reference_triples += kg_neighborhood_extractor(
                    rdf_path=f"{KNOWLEDGE_GRAPH_PATH}/{DATASET_NAME}_ontology_b{row['batch_num']}_q{row['question_num']}.rdf",
                    per=0.75, n=2, key=triple[0], k=3, embedding_model="text-embedding-3-small"
                )

            if len(triple[2]) > 1:
                reference_triples += kg_neighborhood_extractor(
                    rdf_path=f"{KNOWLEDGE_GRAPH_PATH}/{DATASET_NAME}_ontology_b{row['batch_num']}_q{row['question_num']}.rdf",
                    per=0.75, n=2, key=triple[2], k=3, embedding_model="text-embedding-3-small"
                )

        batch_input_item.append({
            "question": row['question'],
            "reference_triples": reference_triples
        })
    batch_input_list.append(batch_input_item)
    break

Loading RDF graph...
Extracting entities...
Total entities found: 148
Loading existing embeddings from 'embedding_dict.csv'...
Initializing LangChain OpenAIEmbeddings with model 'text-embedding-3-small'...
Getting embedding for search key: 'arthurs_magazine'
Embedding for 'arthurs_magazine' found in cache.
[Thread 1] started.
Embedding for 'arthurs_magazine' found in cache.
Embedding for 'bengaluru' found in cache.
Embedding for 'a_day_later' found in cache.
Embedding for 'n2014_year-end' found in cache.
Embedding for 'may_1846' found in cache.
Embedding for 'echosmith' found in cache.
Embedding for 'first_college_established_specifically_for_women_in_south' found in cache.
Embedding for 'bauer_media_group' found in cache.
Embedding for 'perhaps_the_smallest_courthouse_in_united_states' found in cache.
Embedding for 'music_videos' found in cache.
Embedding for 'bachelors_degrees' found in cache.
Embedding for 'santa_ana_canyon' found in cache.
Embedding for 'justin_timberlake' found in

In [45]:
# convert '_' into ' ' and remove first 'n' if second charactor is an integer of every entity or relationship of reference_triples
def format_reference_triples(reference_triples):
    formatted_triples = []
    for triple in reference_triples:
        subject, relation, obj = triple
        subject = subject.replace('_', ' ')
        relation = relation.replace('_', ' ')
        obj = obj.replace('_', ' ')
        if subject.startswith('n') and subject[1].isdigit():
            subject = subject[1:]
        if obj.startswith('n') and obj[1].isdigit():
            obj = obj[1:]
        formatted_triples.append((subject, relation, obj))
    return formatted_triples

for i, batch in enumerate(batch_input_list):
    print(f"Batch {i} ... processing {len(batch)} items")
    for j, item in enumerate(batch):
        item['reference_triples'] = format_reference_triples(item['reference_triples'])
        print(f"Item {j}:")
        print("Question:", item['question'])
        print("Reference Triples:", item['reference_triples'])
        print("-" * 50)

Batch 0 ... processing 10 items
Item 0:
Question: Which magazine was started first Arthur's Magazine or First for Women?
Reference Triples: [('arthurs magazine', 'was edited by', 'ts arthur'), ('arthurs magazine', 'merged in', 'may 1846'), ('arthurs magazine', 'was published during', '1844 1846'), ('arthurs magazine', 'was a', 'literary periodical'), ('arthurs magazine', 'was published in', 'philadelphia'), ('arthurs magazine', 'merged into', 'godeys ladys book'), ('arthurs magazine', 'featured work by', 'jh ingraham'), ('arthurs magazine', 'featured work by', 'sarah josepha hale'), ('william rast', 'is located in', 'united states'), ('arthurs magazine', 'was published in', 'united states'), ('arthurs magazine', 'featured work by', 'edgar a poe'), ('arthurs magazine', 'featured work by', 'thomas g spear'), ('first arthur county courthouse and jail', 'is located in', 'united states'), ('echosmith', 'is located in', 'united states'), ('first for women', 'is published in', 'united states'

## 6. Getting the final answer

In [51]:
system_msg = """
<role>
    You are a logical reasoning expert tasked with answering questions based only on provided structured knowledge in the form of Reference Triples.
</role>

<behavior>
    <rule name="Step-by-Step Reasoning">
        Use the reference triples to deduce relevant facts in a step-by-step manner. Clearly identify which triples are used for each inference.
    </rule>
    <rule name="Comparative Reasoning">
        When comparing entities, extract or infer comparable attributes (e.g., start dates, founding years) and use them to conclude which comes first, is larger, etc., as applicable.
    </rule>
    <rule name="Answer Reporting">
        At the end of your reasoning, provide the final answer on a new line prefixed with "### FINAL_ANSWER", followed by the concise answer on the next line.
        If the data is insufficient to answer the question definitively, return "insufficient data".
    </rule>
</behavior>

<format>
1. Carefully examine the <question> and <reference_triples>.
2. Begin with step-by-step reasoning based strictly on the reference triples.
3. End your output with the heading "### FINAL_ANSWER" followed by the answer on the next line.
</format>
"""

human_msg = """
<question>
{question}
</question>

<reference triples>
{reference_triples}
</reference triples>
"""

prompt = ChatPromptTemplate([("system", system_msg), ("human", human_msg)])

In [52]:
chain = prompt | llm_model | StrOutputParser()

In [53]:
answer_str_batch = []

for i, batch in enumerate(batch_input_list):
    print(f"Batch {i + 1} ... processing {len(batch)} items")
    answer_str_list = chain.batch(batch)
    answer_str_batch.append(answer_str_list)

Batch 1 ... processing 10 items


In [54]:
for i, answer_str_list in enumerate(answer_str_batch):
    print(f"Batch {i + 1} ... processed {len(answer_str_list)} items")
    for j, answer_str in enumerate(answer_str_list):
        print(f"Item {j + 1}:")
        print(answer_str)
        print("-" * 50)

# Arthur's Magazine
# Delhi
# President Richard Nixon
# American
# alcohol
# Jonathan Stark
# Crambidae
# Badr Hari
# 2006
# 6.213 km long

Batch 1 ... processed 10 items
Item 1:
Step-by-Step Reasoning:

1. The question asks which magazine was started first: Arthur's Magazine or First for Women.
2. From the reference triples:
   - ('arthurs magazine', 'was published during', '1844 1846') suggests that Arthur's Magazine started in 1844.
   - ('first for women', 'was started in', '1989') states that First for Women was started in 1989.
3. Comparing these two years:
   - Arthur's Magazine: 1844
   - First for Women: 1989
4. 1844 predates 1989.

### FINAL_ANSWER
Arthur's Magazine was started first.
--------------------------------------------------
Item 2:
Step-by-step reasoning:

1. We need to find the city where the head office of the hotel company that the Oberoi family is part of is located.
2. The triple ('oberoi family', 'is involved in', 'hotels') suggests the Oberoi family is linked to the hotel industry.
3. The triple ('oberoi family', 'is involved with', 'the oberoi group') shows specifically that the Oberoi family i

In [55]:
questions_str_df = pd.DataFrame({
    "batch_num": [],
    "question_num": [],
    "questions_str": []
})

for i, answer_str_list in enumerate(answer_str_batch):
    batch_num_list = [i] * len(answer_str_list)
    question_num_list = list(range(1, len(answer_str_list) + 1))
    questions_str_df = pd.concat([questions_str_df, pd.DataFrame({
        "batch_num": batch_num_list,
        "question_num": question_num_list,
        "questions_str": answer_str_list
    })], ignore_index=True)

questions_str_df

,batch_num,question_num,questions_str
0,0.0,1.0,Step-by-Step Reasoning:\n\n1. The question ask...
1,0.0,2.0,Step-by-step reasoning:\n\n1. We need to find ...
2,0.0,3.0,Step-by-step reasoning:\n\n1. The question ask...
3,0.0,4.0,Step-by-Step Reasoning:\n\n1. The question ask...
4,0.0,5.0,Step-by-step reasoning:\n1. The question asks:...
5,0.0,6.0,Step-by-step reasoning:\n\n1. The question ask...
6,0.0,7.0,Step-by-step reasoning:\n\n1. The question ask...
7,0.0,8.0,Step-by-Step Reasoning:\n\n1. The question ask...
8,0.0,9.0,Step-by-step reasoning:\n\n1. The question ask...
9,0.0,10.0,Step-by-step reasoning:\n\n1. The question ask...
